# CreditGuard — Phase 2: Data Understanding & Inspection

## Project Objective
The primary objective of **CreditGuard** is to analyze customer credit card repayment behavior, identify high-risk customer segments, predict the probability of default, and present actionable financial insights through interactive Power BI dashboards and a Streamlit web application.

## Dataset Source Description
This notebook analyzes the **UCI Default of Credit Card Clients Dataset**.
- **Source:** UCI Machine Learning Repository / Taiwan Credit Card Default Dataset
- **Records:** 30,000 customer records
- **Attributes:** 25 columns covering credit limits, demographic information, 6-month repayment status history, monthly bill amounts, previous payment amounts, and default target status.

## Beginner-Friendly Data Dictionary

| Feature Name | Type | Description | Values / Range |
|---|---|---|---|
| `ID` | Integer | Unique identifier for each credit card client | 1 to 30,000 |
| `LIMIT_BAL` | Float | Amount of given credit in NT dollars (individual + family credit) | 10,000 to 1,000,000 |
| `SEX` | Integer | Gender of the customer | `1` = Male, `2` = Female |
| `EDUCATION` | Integer | Highest education level attained | `1` = Graduate School, `2` = University, `3` = High School, `4` = Others, `0`/`5`/`6` = Undocumented |
| `MARRIAGE` | Integer | Marital status | `1` = Married, `2` = Single, `3` = Others, `0` = Undocumented |
| `AGE` | Integer | Customer age in years | 21 to 79 |
| `PAY_0` to `PAY_6` | Integer | Repayment status from September (`PAY_0`) to April (`PAY_6`) 2005 | `-2` = Duly paid/No consumption, `-1` = Paid in full, `0` = Revolving credit used, `1` to `8` = Payment delay in months |
| `BILL_AMT1` to `BILL_AMT6` | Float | Monthly bill statement amount in NT dollars (Sept to April 2005) | Real numerical values (includes negative bill balances) |
| `PAY_AMT1` to `PAY_AMT6` | Float | Amount paid in previous month in NT dollars (Sept to April 2005) | Real numerical values |
| `default_payment_next_month` | Integer | Target variable indicating default payment in October 2005 | `1` = Defaulted, `0` = Did not default |

## 1. Environment Setup & Safe Dataset Loading

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src folder to python path for modular import
sys.path.append(os.path.abspath("../src"))
from data_cleaning import load_raw_data, validate_dataset

# Load dataset using project-relative path
df = load_raw_data("../data/raw/UCI_Credit_Card.csv")
validate_dataset(df)

## 2. Dataset Dimensions & Sample Data Preview

In [ ]:
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
print("--- First 5 Rows ---")
display(df.head())

print("\n--- Last 5 Rows ---")
display(df.tail())

## 3. Column Listing & Data Types Inspection

In [ ]:
col_summary = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values
})
display(col_summary)

## 4. Missing-Value, Duplicate-Row & Duplicate-ID Inspection

In [ ]:
total_nulls = df.isnull().sum().sum()
total_dup_rows = df.duplicated().sum()
unique_ids = df['ID'].nunique()
has_dup_ids = df['ID'].duplicated().any()

print(f"Total Missing Values across Dataset: {total_nulls}")
print(f"Total Duplicate Rows:               {total_dup_rows}")
print(f"Number of Unique Customer IDs:     {unique_ids:,}")
print(f"Duplicate Customer IDs Exist:        {'Yes' if has_dup_ids else 'No'}")

## 5. Categorical & Repayment-Status Inspection
Inspecting value distributions for `SEX`, `EDUCATION`, `MARRIAGE`, and repayment history columns (`PAY_0` to `PAY_6`).

In [ ]:
print("=== Demographic Categorical Value Distributions ===\n")
for col in ['SEX', 'EDUCATION', 'MARRIAGE']:
    print(f"--- Column '{col}' ---")
    print(df[col].value_counts(dropna=False))
    print()

print("=== Repayment Status Value Distributions (PAY_0 to PAY_6) ===\n")
for col in ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']:
    print(f"--- Column '{col}' ---")
    print(df[col].value_counts(dropna=False).sort_index().to_dict())
    print()

## 6. Target Class Distribution (`default_payment_next_month`)

In [ ]:
target_col = "default_payment_next_month"
counts = df[target_col].value_counts()
percentages = df[target_col].value_counts(normalize=True) * 100

target_table = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages.round(2)
})
display(target_table)

# Simple Target Distribution Chart (Allowed)
plt.figure(figsize=(6, 4))
bars = plt.bar(['Non-Defaulter (0)', 'Defaulter (1)'], counts, color=['#2ca02c', '#d62728'], width=0.5)
plt.title('Target Variable Distribution (default_payment_next_month)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Customers', fontsize=10)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 300, f'{height:,} ({height/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
plt.ylim(0, 26000)
plt.tight_layout()
plt.show()

## 7. Initial Data-Quality Observations

1. **Target Class Imbalance:** Defaulters represent **22.12%** (6,636 clients) vs non-defaulters at **77.88%** (23,364 clients). This moderate imbalance will require techniques such as SMOTE, undersampling, or class-weight adjustments in Phase 5.
2. **Data Completeness & ID Integrity:** 0 missing values, 0 duplicate rows, and exactly 30,000 unique `ID` values.
3. **Undocumented Categorical Values:**
   - `EDUCATION`: Contains undocumented categories `0` (14), `4` (123), `5` (280), and `6` (51).
   - `MARRIAGE`: Contains undocumented category `0` (54).
4. **Repayment History Encoding:** `PAY_0` through `PAY_6` use `-2` (no consumption) and `0` (revolving credit), which should be documented or mapped clearly during feature engineering.
5. **Negative Bill Amounts:** `BILL_AMT` variables contain negative numbers representing customer credit balances / overpayments.